In [ ]:
/_static/db/wilrijk.db

# Detective: de spookschade

:::{admonition} Leerdoelen
:class: tip

Ook in deze les los je een fraudeonderzoek op met SQL. Nieuwe leerstof is er niet: je combineert alles wat je kan, van SELECT tot HAVING.

Na deze les kan je:
- met GROUP BY en aggregatiefuncties patronen vinden die je in losse rijen niet ziet
- met HAVING verdachte groepen uit een resultaat filteren
- de juiste aggregatiefunctie kiezen: tellen (COUNT) is niet hetzelfde als optellen (SUM)
:::

## Het dossier

Na de zaak-Steenberg heeft je reputatie zich rondgesproken. Dit keer belt **Adventure Works** — de fietsenbouwer die je intussen goed kent uit de toepassingslessen. Hun Europese distributiecentrum in **Wilrijk** heeft een probleem.

Bij het afsluiten van het derde kwartaal zag het hoofdkantoor iets vreemds in de cijfers: het magazijn in Wilrijk boekte voor een klein fortuin aan voorraad af wegens schade. Beschadigde spullen afboeken is normaal — een gescheurd truitje hier, een lekke band daar — maar niet voor zulke bedragen.

Deze keer is er geen ooggetuige en geen alibi om te kraken. De fraude verstopt zich tussen tientallen gewone, onschuldig ogende rijen. Je vindt ze alleen door te tellen en op te tellen — precies waar GROUP BY en HAVING voor dienen.

## De database

Uit AdventureWorks ken je deze tabellen al:

- **Product**(ProductID, Name, ProductNumber, Color, StandardCost, ListPrice, ProductCategoryID, ...)
- **ProductCategory**(ProductCategoryID, ParentProductCategoryID, Name)

Nieuw zijn de tabellen van het distributiecentrum in Wilrijk:

- **magazijnmedewerkers**(medewerker_id, voornaam, achternaam, functie, ploeg, gsm)
- **afboekingen**(afboeking_id, datum, product_id, aantal, reden, medewerker_id) — het logboek van het derde kwartaal: elke keer dat iemand voorraad afboekte (beschadigd, defect, niet verkoopbaar)
- **memos**(memo_id, datum, afzender, onderwerp, tekst)
- **verhoren**(verhoor_id, medewerker_id, datum, tekst)
- **zoekertjes**(zoekertje_id, verkoper, omschrijving, prijs, datum, gsm) — alle fietszoekertjes die de interne audit verzamelde op de tweedehandssite Tweedewiel.be
- **controle**(verdachte, uitkomst) — hier controleer je op het einde je antwoord

:::{admonition} Spelregels
:class: warning
- Werk in duo's: de ene typt, de andere denkt mee en noteert wat jullie vinden.
- Elke stap levert een nieuw spoor op: een bedrag, een naam, een nummer. **Schrijf die op**, je hebt ze in de volgende stappen nodig.
- Zit je vast? Onder elke stap staat een hint. Klap die pas open als je het echt zelf geprobeerd hebt.
- En uiteraard: een echte detective kijkt niet stiekem in de tabel `controle`. Die is alleen voor je eindantwoord.
:::

## Stap 1: de memo

Alles begint met een memo van het hoofdkantoor aan het magazijn, verstuurd op **6 oktober 2025**. Vraag ze op — en lees ze aandachtig.

In [ ]:
--- Stap 1: vraag de memo van 6 oktober 2025 op.

:::{admonition} Hint bij stap 1
:class: tip dropdown
Filter de tabel `memos` op datum. Datums zijn tekst in het formaat `YYYY-MM-DD`.
``` SQL
SELECT afzender, onderwerp, tekst
FROM memos
WHERE datum = '...';
```
:::

## Stap 2: hoe groot is het probleem?

De memo noemt een totaalbedrag. Een detective gelooft niets zonder het zelf na te rekenen — en wil vooral weten **waar** dat bedrag zit.

Bereken per categorie de totale afgeboekte waarde. De waarde van één afboeking is `aantal * StandardCost` (wat het product het bedrijf kostte). Je hebt drie tabellen nodig: `afboekingen`, `Product` en `ProductCategory`. Sorteer van groot naar klein.

Wat valt je op? Schade is normaal klein spul...

*Tip: `ROUND(x, 2)` rondt een berekening af op twee cijfers na de komma.*

In [ ]:
--- Stap 2: bereken per categorie de totale afgeboekte waarde, van groot naar klein.

:::{admonition} Hint bij stap 2
:class: tip dropdown
`afboekingen` koppel je aan `Product` via `product_id` en `ProductID`, en `Product` aan `ProductCategory` via `ProductCategoryID`.
``` SQL
SELECT c.Name, ROUND(SUM(a.aantal * p.StandardCost), 2) AS waarde
FROM afboekingen a
JOIN Product p ON a.product_id = p.ProductID
JOIN ProductCategory c ON p.ProductCategoryID = c.ProductCategoryID
GROUP BY c.Name
ORDER BY waarde DESC;
```
Wil je het bedrag uit de memo exact terugzien? Laat de `GROUP BY` en de categorie weg: dan berekent `SUM` de totale waarde in één keer.
:::

## Stap 3: wie boekt er zoveel af?

Bijna het hele bedrag zit bij twee categorieën **fietsen** — terwijl schade normaal uit banden, truitjes en ander klein spul bestaat. Er wordt dus afgeboekt wat niet afgeboekt hoort te worden.

Elke afboeking is geregistreerd door een medewerker. Tel het aantal afboekingen per medewerker en toon er meteen de naam en functie bij. Toon enkel wie **20 of meer** afboekingen deed.

Noteer ook de `medewerker_id`: die heb je straks nodig.

In [ ]:
--- Stap 3: wie deed 20 of meer afboekingen dit kwartaal?

:::{admonition} Hint bij stap 3
:class: tip dropdown
Groepeer per medewerker en filter de groepen met `HAVING`.
``` SQL
SELECT m.medewerker_id, m.voornaam, m.achternaam, m.functie,
       COUNT(*) AS aantal_afboekingen
FROM afboekingen a
JOIN magazijnmedewerkers m ON a.medewerker_id = m.medewerker_id
GROUP BY m.medewerker_id
HAVING COUNT(*) >= 20;
```
:::

## Stap 4: het verhoor

Eén naam springt eruit: iemand van de retourafdeling, met veel meer afboekingen dan alle anderen. Verdacht? Voor je conclusies trekt: lees zijn verklaring in de tabel `verhoren`.

Hij geeft je — misschien zonder het te beseffen — een gouden tip.

In [ ]:
--- Stap 4: lees het verhoor van de medewerker met de meeste afboekingen.

:::{admonition} Hint bij stap 4
:class: tip dropdown
``` SQL
SELECT tekst
FROM verhoren
WHERE medewerker_id = ...;
```
Lees goed: wat moet je volgens hem tellen — en wat net niet?
:::

## Stap 5: tellen is niet optellen

Hij heeft een punt. Wie de retours afhandelt, boekt nu eenmaal váák af — dat bewijst niets. Hoe vaak iemand afboekt is hier de verkeerde maatstaf. Wat al die afboekingen samen **waard** zijn, dát telt.

Herhaal daarom stap 3, maar bereken per medewerker de totale afgeboekte waarde (`aantal * StandardCost`). Toon enkel de medewerkers met **meer dan 5000 euro**.

In [ ]:
--- Stap 5: wie boekte voor meer dan 5000 euro af? Toon ook het aantal afboekingen.

:::{admonition} Hint bij stap 5
:class: tip dropdown
Dezelfde opbouw als stap 3, maar nu met `SUM` in plaats van `COUNT` — en een extra `JOIN` op `Product` voor de prijs.
``` SQL
SELECT m.medewerker_id, m.voornaam, m.achternaam,
       COUNT(*) AS aantal,
       ROUND(SUM(a.aantal * p.StandardCost), 2) AS waarde
FROM afboekingen a
JOIN Product p ON a.product_id = p.ProductID
JOIN magazijnmedewerkers m ON a.medewerker_id = m.medewerker_id
GROUP BY m.medewerker_id
HAVING SUM(a.aantal * p.StandardCost) > 5000;
```
Vergelijk met stap 3: de meeste afboekingen en de grootste waarde — dat is niet dezelfde persoon.
:::

## Stap 6: het patroon

Een heel andere naam — iemand met maar een handvol afboekingen, samen toch goed voor een klein fortuin. Bekijk zijn afboekingen stuk voor stuk: welke producten, hoeveel per keer, met welke reden, en wanneer?

Lees daarna drie verhoren: dat van je verdachte, dat van de magazijnverantwoordelijke en dat van de teamleider van de avondploeg.

In [ ]:
--- Stap 6a: bekijk alle afboekingen van je nieuwe verdachte, van oud naar nieuw.

--- Stap 6b: lees de verhoren van je verdachte, de magazijnverantwoordelijke en de teamleider van de avondploeg.

:::{admonition} Hint bij stap 6
:class: tip dropdown
``` SQL
SELECT a.datum, p.Name, a.aantal, a.reden, p.StandardCost
FROM afboekingen a
JOIN Product p ON a.product_id = p.ProductID
WHERE a.medewerker_id = ...
ORDER BY a.datum;
```
De `medewerker_id` van de magazijnverantwoordelijke en de teamleider van de avondploeg zoek je op in `magazijnmedewerkers`. Met `IN` haal je de drie verhoren in één keer op:
``` SQL
SELECT m.voornaam, m.achternaam, v.tekst
FROM verhoren v
JOIN magazijnmedewerkers m ON v.medewerker_id = m.medewerker_id
WHERE v.medewerker_id IN (..., ..., ...);
```
Let op het patroon: altijd hetzelfde soort product, altijd dezelfde reden, altijd één stuk per keer, netjes gespreid over het kwartaal. Wie steelt in schijfjes, valt in geen enkele losse rij op — alleen de som verraadt hem.
:::

## Stap 7: de zoekertjes

Volgens zijn verhoor was elke fiets total loss en ging alles de schrootcontainer in. Maar de magazijnverantwoordelijke zei iets opvallends: bij grote transportschade hoort een verzekeringsclaim — en de vervoerder kreeg er dit kwartaal amper.

De interne audit verzamelde intussen alle fietszoekertjes van de voorbije maanden op Tweedewiel.be, mét het gsm-nummer dat bij elk zoekertje staat. Drie vragen:

1. Welke verkoper plaatste opvallend veel zoekertjes — zeg maar **5 of meer**?
2. Wat verkoopt die persoon precies, en in welke staat?
3. Wie zit er achter die schuilnaam? Eén kolom in `zoekertjes` komt ook voor in `magazijnmedewerkers`...

In [ ]:
--- Stap 7a: welke verkoper plaatste 5 of meer zoekertjes?

--- Stap 7b: bekijk alle zoekertjes van die verkoper.

--- Stap 7c: wie is het? Koppel zoekertjes aan magazijnmedewerkers.

:::{admonition} Hint bij stap 7
:class: tip dropdown
``` SQL
SELECT verkoper, COUNT(*) AS aantal
FROM zoekertjes
GROUP BY verkoper
HAVING COUNT(*) >= 5;
```
``` SQL
SELECT omschrijving, prijs, datum
FROM zoekertjes
WHERE verkoper = '...';
```
``` SQL
SELECT DISTINCT z.verkoper, m.voornaam, m.achternaam, m.functie
FROM zoekertjes z
JOIN magazijnmedewerkers m ON z.gsm = m.gsm;
```
Gloednieuwe topfietsen, "doos nog dicht" — van modellen die zogezegd total loss waren. En kijk ook eens naar de cijfers in de schuilnaam: 2610 is de postcode van Wilrijk.
:::

## De ontknoping

Je hebt nu alles: de **cijfers** (één medewerker boekte voor tienduizenden euro's aan topfietsen af), het **patroon** (altijd één stuk, altijd 'transportschade', nooit een claim bij de vervoerder) en het **bewijs** (dezelfde modellen staan gloednieuw te koop — met zijn gsm-nummer erbij).

Wijs je dader aan in de tabel `controle`. Vul de volledige naam in, precies zoals die in `magazijnmedewerkers` staat:

``` SQL
SELECT uitkomst
FROM controle
WHERE verdachte = 'Voornaam Achternaam';
```

In [ ]:
--- Wie pleegde de fraude? Controleer je antwoord in de tabel controle.

:::{admonition} Zaak gesloten?
:class: tip
Kreeg je *"Juist!"* te zien? Proficiat, detective. Overloop dan samen nog even het dossier:

- In stap 3 wees `COUNT(*)` naar de verkeerde persoon, in stap 5 wees `SUM(...)` naar de dader. Hoe kan dat — en wat leert je dat over het kiezen van een aggregatiefunctie?
- De fraudeur werkte "in schijfjes": nooit meer dan één fiets tegelijk. Waarom is net daarom een samenvatting per groep (GROUP BY) het enige wapen dat werkt?
- Welke afspraak had deze fraude kunnen voorkomen? Denk aan een tweede handtekening boven een bepaald bedrag, een verplichte schadeclaim bij de vervoerder, of een kwartaalrapport zoals dat van stap 5.
:::